In [1]:
from ml_recon.Models.Unet import Unet
from torch.utils.data import DataLoader
from ml_recon.Transforms import (pad, combine_coil, toTensor, addChannels, 
                                   view_as_real, fft_2d, normalize, pad_recon)
from ml_recon.Dataset.undersampled_dataset import UndersampledKSpaceDataset
from torchvision.transforms import Compose
import numpy as np
import torch
from ml_recon.Utils.collate_function import collate_fn
from datetime import datetime
from ml_recon.Utils.save_model import save_model

In [2]:
torch.manual_seed(0)
np.random.seed(0)

In [3]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter('/home/kadotab/scratch/runs/' +  datetime.now().strftime("%Y%m%d-%H%M%S"))

In [4]:
%load_ext autoreload
%autoreload 2 

In [4]:
transforms = Compose(
    (
        pad((640, 320)),
        pad_recon((320, 320)), 
        fft_2d(axes=[-1, -2]),
        combine_coil(),
        toTensor(),
        normalize(), 
        addChannels(),
        view_as_real(),
    )
)
dataset = UndersampledKSpaceDataset('/home/kadotab/projects/def-mchiew/kadotab/Datasets/fastMRI/multicoil_train', transforms=transforms, R=2)
dataloader = DataLoader(dataset, batch_size=1, collate_fn=collate_fn)
    

In [13]:
data = next(iter(dataloader))

MemoryError: Unable to allocate 875. MiB for an array with shape (14, 20, 320, 640) and data type complex128

In [5]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [6]:
model = Unet(2, 2)
model.to(device)

Unet(
  (down_sample_layers): ModuleList(
    (0): double_conv(
      (conv1): Conv2d(2, 18, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (conv2): Conv2d(18, 18, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (activation): LeakyReLU(negative_slope=0.2, inplace=True)
      (instance_norm1): InstanceNorm2d(18, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (instance_norm2): InstanceNorm2d(18, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (drop_out1): Dropout2d(p=0, inplace=False)
      (drop_out2): Dropout2d(p=0, inplace=False)
    )
    (1): Unet_down(
      (down): down(
        (max_pool): AvgPool2d(kernel_size=2, stride=(2, 2), padding=0)
      )
      (conv): double_conv(
        (conv1): Conv2d(18, 36, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (conv2): Conv2d(36, 36, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (activation): LeakyRe

In [8]:
checkpoint = torch.load('/home/kadotab/python/ml/ml_recon/Model_Weights/20230418-181325Unet.pt', map_location=torch.device('cpu'))

In [10]:
model.load_state_dict(checkpoint['model'])

<All keys matched successfully>

In [14]:
output = model(data['undersampled'][[5], :, :])

NameError: name 'data' is not defined

In [15]:
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [16]:
model_weight_path = '/home/kadotab/python/ml/ml_recon/Model_Weights/'
def train(model, loss_function, optimizer, dataloader, epoch=7):
    cur_loss = 0
    current_index = 0
    try:
        for e in range(epoch):
            for data in dataloader:
                undersampled = data['undersampled']
                recon = data['recon']
                for i in range(undersampled.shape[0]):
                    optimizer.zero_grad()

                    recon_slice = torch.flip(recon[[i], ...].float(), (1, ))
                    undersampled_slice = undersampled[[i],...].float()

                    recon_slice = recon_slice.to(device)
                    undersampled_slice = undersampled_slice.to(device)
    
                    predicted_sampled = model(undersampled_slice)

                    predicted_sampled = torch.sqrt(predicted_sampled.pow(2).sum(1)+ 1e-6)
                    predicted_sampled = predicted_sampled[:, 160:-160, :]
                    loss = loss_function(predicted_sampled, recon_slice)
                    
                    loss.backward()
                    optimizer.step()
                    
                    cur_loss += loss.item()
                    current_index += 1
                    if current_index % 1000 == 999:
                        writer.add_histogram('sens/weights1', next(model.down_sample_layers[0].conv1.parameters()), current_index)
                        writer.add_histogram('castcade0/weights1', next(model.down_sample_layers[0].conv2.parameters()), current_index)
                        writer.add_histogram('castcade0/weights2', next(model.down_sample_layers[3].conv.conv1.parameters()), current_index)
                        writer.add_histogram('castcade0/weights11', next(model.up_sample_layers[3].conv.conv2.parameters()), current_index)
                        writer.add_histogram('castcade0/weights12', next(model.conv1d.parameters()), current_index)
                        writer.add_scalar('Loss/train', cur_loss, current_index)
                        print(f"Iteration: {current_index + 1:>d}, Loss: {cur_loss:>7f}")
                        save_model(model_weight_path, model, optimizer, current_index)
                    cur_loss = 0
    except KeyboardInterrupt:
        pass

    save_model(model_weight_path, model, optimizer, current_index)

In [17]:
train(model, loss_fn, optimizer, dataloader)